# Play with Docker Provider

本文档演示如何使用 Docker Runtime Provider 在本地 Docker 环境中管理沙箱容器。 Docker Runtime Provider 适合本地开发和调试场景。

## 初始化环境

首先导入必要的模块并配置日志，以便查看 Runtime 的详细操作过程。

In [1]:
from nexus.utils.log import setup_logging
from nexus.runtime import DockerRuntimeProvider

setup_logging()

## 创建 Runtime Provider

创建 `DockerRuntimeProvider` 实例。 可以配置主容器镜像、Runtime 镜像等参数。

**注意**：首次运行时，会自动构建 `nexus-runtime` 镜像，这可能需要几分钟时间。

In [2]:
rp = DockerRuntimeProvider()

## 获取 Runtime 实例

调用 `acquire()` 方法创建并启动一个新的 Docker 容器作为 Runtime 实例。 需要指定：
- `name`：Runtime 实例的唯一名称
- `image`：要运行的主容器镜像

该方法会自动：
1. 确保 Runtime 镜像已构建
2. 创建共享的 Runtime 容器（提供 /nix 环境）
3. 启动用户容器并挂载 /nix 卷
4. 等待容器就绪并返回 `Runtime` 对象

In [3]:
runtime = await rp.acquire(name='main', image='ubuntu:24.04')

2026-01-13 01:24:34 | INFO | nexus.runtime.docker | Image nexus-runtime:0.8.6 already exists, skipping build
2026-01-13 01:24:34 | INFO | nexus.runtime.docker | Runtime container nexus-runtime-0-8-6 already exists
2026-01-13 01:24:34 | INFO | nexus.runtime.docker | Starting container nexus-user-main...
2026-01-13 01:24:43 | INFO | nexus.runtime.docker | Runtime main ready at http://172.17.0.1:33154


## 在 Runtime 中执行命令

使用 `runtime.run_command()` 方法在容器中执行命令。 这里执行 `cat /etc/os-release` 来查看容器的操作系统信息。

命令执行结果包含标准输出（`stdout`）、标准错误（`stderr`）和退出码（`exit_code`）。

In [4]:
result = await runtime.run_command('cat /etc/os-release')

print(result.stdout)

2026-01-13 01:24:43 | INFO | nexus.core.registry | Imported 'nexus.extensions.agents.demo.tools' with 2 registrations:
2026-01-13 01:24:43 | INFO | nexus.core.registry |   - [tool] demo/think, demo/bash
2026-01-13 01:24:43 | INFO | nexus.core.registry | Imported 'nexus.extensions.agents.swe_agent.tools' with 32 registrations:
2026-01-13 01:24:43 | INFO | nexus.core.registry |   - [executor] create_browser_session, destroy_browser_session, browser_goto, browser_go_back, browser_go_forward, browser_click, browser_fill, browser_scroll, browser_press_key, browser_hover, browse [... (truncated log)] l_session, run_command_in_terminal_session, start_command_in_terminal_session, query_terminal_command_status, send_keys_to_terminal_session, capture_terminal_pane, terminate_terminal_session_processes
2026-01-13 01:24:43 | INFO | nexus.core.registry |   - [tool] swe_agent/bash, swe_agent/str_replace_editor, swe_agent/review_submit
2026-01-13 01:24:44 | INFO | nexus.core.registry | Imported 'nexu

## 验证 /nix 环境

Docker Runtime 通过挂载共享的 Runtime 容器提供 /nix 环境。 我们可以验证 Python 环境是否可用。

In [5]:
result = await runtime.run_command('/nix/runtime/nix-env/bin/python --version')

print(result.stdout)

Python 3.10.19



## 释放所有 Runtime

使用 `release_all()` 方法停止并删除所有创建的用户容器。

对于单个 Runtime 实例，也可以使用 `release(name)` 方法单独释放。

**注意**：这不会删除共享的 Runtime 容器。 如需完全清理，可以调用 `cleanup_runtime_container()` 方法。

In [6]:
await rp.release_all()

2026-01-13 01:24:45 | INFO | nexus.runtime.docker | Stopping container nexus-user-main...
2026-01-13 01:24:45 | INFO | nexus.runtime.docker | Runtime main released
